In [30]:
import pandas as pd
from pathlib import Path

def load_data(file_path):
    """
    Load data from a CSV file.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    try:
        data = pd.read_csv(file_path, sep="|")
        # data = pd.read_json(file_path, lines=True)
        print(f"Data loaded successfully from {file_path}")
        return data
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return None

filepath = Path("/home/user/Downloads/2026-07-10/DataHut_SI_Mercator_FullDump_20260709.CSV")   
loaded_data = load_data(filepath)

print("\nDATASET SHAPE")
print("--------------------------------")
print("The shape =", loaded_data.shape)

Data loaded successfully from /home/user/Downloads/2026-07-10/DataHut_SI_Mercator_FullDump_20260709.CSV

DATASET SHAPE
--------------------------------
The shape = (16871, 132)


/tmp/ipykernel_6301/2013168749.py:15: DtypeWarning: Columns (95) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, sep="|")


In [31]:
loaded_data.columns.tolist()

['unique_id',
 'competitor_name',
 'store_name',
 'store_addressline1',
 'store_addressline2',
 'store_suburb',
 'store_state',
 'store_postcode',
 'store_addressid',
 'extraction_date',
 'product_name',
 'brand',
 'brand_type',
 'grammage_quantity',
 'grammage_unit',
 'drained_weight',
 'producthierarchy_level1',
 'producthierarchy_level2',
 'producthierarchy_level3',
 'producthierarchy_level4',
 'producthierarchy_level5',
 'producthierarchy_level6',
 'producthierarchy_level7',
 'regular_price',
 'selling_price',
 'price_was',
 'promotion_price',
 'promotion_valid_from',
 'promotion_valid_upto',
 'promotion_type',
 'percentage_discount',
 'promotion_description',
 'package_sizeof_sellingprice',
 'per_unit_sizedescription',
 'price_valid_from',
 'price_per_unit',
 'multi_buy_item_count',
 'multi_buy_items_price_total',
 'currency',
 'breadcrumb',
 'pdp_url',
 'variants',
 'product_description',
 'instructions',
 'storage_instructions',
 'preparationinstructions',
 'instructionforuse',


In [3]:
loaded_data.store_name.unique()


array([nan])

In [12]:
column_list = ['unique_id','competitor_name','extraction_date','product_name','grammage_quantity','grammage_unit','regular_price','selling_price','price_valid_from','price_per_unit','percentage_discount','promotion_price','promotion_valid_from','promotion_valid_upto','promotion_description','currency','breadcrumb','pdp_url','region','pack_size','site_shown_uom']
print(len(column_list))
print(len(loaded_data.columns))

21
32


In [11]:
# Columns in column_list but NOT in loaded_data.columns
missing_columns = set(column_list) - set(loaded_data.columns)
print("Columns in column_list but NOT in loaded_data.columns:")
print(sorted(missing_columns))
print(f"Count: {len(missing_columns)}\n")

# Columns in loaded_data.columns but NOT in column_list
extra_columns = set(loaded_data.columns) - set(column_list)
print("Columns in loaded_data.columns but NOT in column_list:")
print(sorted(extra_columns))
print(f"Count: {len(extra_columns)}")

Columns in column_list but NOT in loaded_data.columns:
[]
Count: 0

Columns in loaded_data.columns but NOT in column_list:
[]
Count: 0


In [32]:
# CSV null summary

null_summary = pd.DataFrame({
    'Null_Count': loaded_data.isnull().sum(),
    'Null_Percentage': (loaded_data.isnull().sum() / len(loaded_data)) * 100
})

# Round percentage to 2 decimal places
null_summary['Null_Percentage'] = null_summary['Null_Percentage'].round(2)

print(null_summary)

# Print column names with 100% null values
completely_null_columns = null_summary.index[null_summary['Null_Percentage'] == 100].tolist()
print("\nColumns with 100% null values:")
print(completely_null_columns)


                    Null_Count  Null_Percentage
unique_id                    0              0.0
competitor_name              0              0.0
store_name               16871            100.0
store_addressline1       16871            100.0
store_addressline2       16871            100.0
...                        ...              ...
suitable_for             16871            100.0
standard_drinks          16871            100.0
environmental            16871            100.0
grape_variety            16871            100.0
retail_limit             16871            100.0

[132 rows x 2 columns]

Columns with 100% null values:
['store_name', 'store_addressline1', 'store_addressline2', 'store_suburb', 'store_state', 'store_postcode', 'store_addressid', 'brand_type', 'drained_weight', 'producthierarchy_level5', 'producthierarchy_level6', 'producthierarchy_level7', 'promotion_valid_from', 'promotion_valid_upto', 'promotion_type', 'package_sizeof_sellingprice', 'per_unit_sizedescription', 'pr

In [9]:
violations = []

for col in loaded_data.columns:
    for idx, value in loaded_data[col].items():
        if value == {} or value == []:
            violations.append({
                "row_index": idx,
                "column": col,
                "value": value
            })

violations_df = pd.DataFrame(violations)

print(f"Number of cells containing {{}} or []: {len(violations_df)}")
print(violations_df)

Number of cells containing {} or []: 0
Empty DataFrame
Columns: []
Index: []


In [13]:

uom_values = [
    "Waschgänge",
    "wg"
]

# Normalize for case-insensitive comparison
uom_values = [x.lower() for x in uom_values]

# Find violations
violations = loaded_data[
    loaded_data["site_shown_uom"]
        .fillna("")
        .str.lower()
        .isin(uom_values)
    &
    (
        loaded_data["grammage_unit"]
            .fillna("")
            .str.lower()
            != "wg"
    )
]

print(f"Number of violations: {len(violations)}")

print(
    violations[
        ["product_name", "site_shown_uom", "grammage_unit"]
    ])

Number of violations: 0
Empty DataFrame
Columns: [product_name, site_shown_uom, grammage_unit]
Index: []


In [37]:
# description column validation

import re

def is_clean_description(text):
    if pd.isna(text):
        return True  # Ignore nulls (change if nulls are invalid)

    text = str(text)

    # # 1. Leading/trailing spaces
    # if text != text.strip():
    #     return False

    # 2. Multiple consecutive spaces
    if re.search(r'\s{2,}', text):
        return False

    # 3. HTML tags
    if re.search(r'<[^>]+>', text):
        return False

    # 4. HTML entities
    if re.search(r'&[a-zA-Z]+;', text):
        return False

    # # 5. Unwanted special characters
    # # Allows letters, numbers, whitespace, and common punctuation
    # if re.search(r'[^A-Za-z0-9\s.,!?()\-\'"/:&%]', text):
    #     return False

    return True

# Find rows with invalid descriptions
invalid_descriptions = loaded_data[
    ~loaded_data['description'].apply(is_clean_description)
]

print(f"Number of invalid descriptions: {len(invalid_descriptions)}")

print(invalid_descriptions[['description']])

Number of invalid descriptions: 0
Empty DataFrame
Columns: [description]
Index: []


In [38]:
import re
import pandas as pd

def is_valid_address(address):
    if pd.isna(address):
        return False

    address = str(address)

    # Empty or whitespace-only
    if address.strip() == "":
        return False

    # Leading/trailing spaces
    if address != address.strip():
        return False

    # Multiple consecutive spaces
    if re.search(r"\s{2,}", address):
        return False

    # # Allow only letters (Unicode), spaces, apostrophes, and hyphens
    # if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿĀ-ž' -]+", address):
    #     return False

    return True

# Find invalid addresses
invalid_addresses = loaded_data[
    ~loaded_data["address"].apply(is_valid_address)
]

print(f"Number of invalid address values: {len(invalid_addresses)}")
print(invalid_addresses[["address"]])

Number of invalid address values: 1
                       address
5092  9104 Church Street  #201


In [5]:
import re
import pandas as pd

def is_valid_email(email):
    if pd.isna(email):
        return False

    email = str(email)

    # Empty or whitespace-only
    if email.strip() == "":
        return False

    # Leading/trailing spaces
    if email != email.strip():
        return False

    # No spaces inside the email
    if " " in email:
        return False

    # No consecutive dots
    if ".." in email:
        return False

    # Email format
    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    return bool(re.fullmatch(pattern, email))

# Find invalid emails
invalid_emails = loaded_data[
    ~loaded_data["email"].apply(is_valid_email)
]

print(f"Number of invalid emails: {len(invalid_emails)}")
print(invalid_emails[["email"]])

Number of invalid emails: 0
Empty DataFrame
Columns: [email]
Index: []


In [7]:
import re
import pandas as pd

def is_valid_country(value):
    if pd.isna(value):
        return False

    value = str(value).strip()

    # Empty check
    if value == "":
        return True

    # Must NOT contain any digits (zipcode/number rule)
    if re.search(r"\d", value):
        return False

    # Allow only letters, spaces, hyphen, apostrophe
    if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ\s\-']+", value):
        return False

    return True

# Find invalid countries
invalid_countries = loaded_data[
    ~loaded_data["country"].apply(is_valid_country)
]

print(f"Number of invalid country values: {len(invalid_countries)}")

print(invalid_countries[["country"]].drop_duplicates())

Number of invalid country values: 0
Empty DataFrame
Columns: [country]
Index: []


In [22]:
expected_prefix = 'https://www.ewm.com/agents/'

# Find rows where profile_url does not start with the expected prefix
invalid_urls = loaded_data[
    ~loaded_data['profile_url']
        .fillna('')
        .astype(str)
        .str.startswith(expected_prefix)
]

print(f"Number of invalid URLs: {len(invalid_urls)}\n")

# Print the invalid URLs
print(invalid_urls['profile_url'])

Number of invalid URLs: 0

Series([], Name: profile_url, dtype: object)


In [30]:
# Find rows where url does not end with id
violations = loaded_data[
    ~loaded_data.apply(
        lambda row: (
            pd.notna(row['url']) and
            pd.notna(row['id']) and
            str(row['url']).rstrip('/').endswith(str(row['id']).strip())
        ),
        axis=1
    )
]

print(f"Number of violations: {len(violations)}\n")

# Print the ids and urls for the violating rows
print(violations[['id', 'url']])

Number of violations: 0

Empty DataFrame
Columns: [id, url]
Index: []


In [42]:
# Find rows where url does not end with id
violations = loaded_data[
    ~loaded_data.apply(
        lambda row: (
            str(row['image_url']).rstrip('/').endswith('webp') or 
            str(row['image_url']).rstrip('/').endswith('jpg')
        ),
        axis=1
    )
]

print(f"Number of violations: {len(violations)}\n")

# Print the ids and urls for the violating rows
print(violations[['profile_url', 'image_url']])

Number of violations: 212

                                             profile_url  \
30     https://www.compass.com/agents/kristina-betanc...   
263         https://www.compass.com/agents/lauren-pepin/   
282          https://www.compass.com/agents/dori-hanson/   
436        https://www.compass.com/agents/diane-farrell/   
691            https://www.compass.com/agents/ryan-shaw/   
...                                                  ...   
32800    https://www.compass.com/agents/cheryl-foote-av/   
34167  https://www.compass.com/agents/fran-flanagan-g...   
34600  https://www.compass.com/agents/the-kim-hamrick...   
34720  https://www.compass.com/agents/the-marlene-bur...   
34824  https://www.compass.com/agents/the-duffy-group...   

                                               image_url  
30     https://www.compass.com/m2/e22bf060-5bfe-44ec-...  
263    https://www.compass.com/m2/e7204ebd-86b2-4b24-...  
282    https://www.compass.com/m2/0c7ed792-b0ac-44df-...  
436    https://w

In [26]:
# Find rows where id is not present in url
violations = loaded_data[
    ~loaded_data.apply(
        lambda row: (
            pd.notna(row['id']) and
            pd.notna(row['url']) and
            str(row['id']).strip() in str(row['url'])
        ),
        axis=1
    )
]

print(f"Number of violations: {len(violations)}\n")

# Print the ids and corresponding urls
print(violations[['id', 'url']])

Number of violations: 0

Empty DataFrame
Columns: [id, url]
Index: []


In [36]:
# Find profile_urls that do NOT contain 'xxxx'
invalid_urls = loaded_data[
    ~loaded_data['profile_url']
        .fillna('')
        .astype(str)
        .str.contains('century21', case=False, na=False)
]

print(f"Number of profile_urls not containing 'century21': {len(invalid_urls)}\n")

# Print only the profile_url values
print(invalid_urls['profile_url'])

Number of profile_urls not containing 'century21': 0

Series([], Name: profile_url, dtype: object)


In [25]:
print(loaded_data[['profile_url', 'image_url']])

                                      profile_url  \
0                 https://AveryHaynes.semonin.com   
1                 https://bonnieroman.semonin.com   
2                 https://braddevries.semonin.com   
3                https://brentlogsdon.semonin.com   
4          https://BridgetteSchroeder.semonin.com   
..                                            ...   
335               https://keithcurran.semonin.com   
336   https://www.semonin.com/bio/jennifercarroll   
337  https://www.semonin.com/bio/lindabakertaylor   
338                https://terryburke.semonin.com   
339               https://zachshively.semonin.com   

                                             image_url  
0    https://content.mediastg.net/Dynamic/RealEstat...  
1    https://content.mediastg.net/Dynamic/RealEstat...  
2    https://content.mediastg.net/Dynamic/RealEstat...  
3    https://content.mediastg.net/Dynamic/RealEstat...  
4    https://content.mediastg.net/Dynamic/RealEstat...  
..                   

In [24]:
def compare_broker_columns(df):
    col_pairs = [
        ("broker", "broker_display_name"),
        ("broker", "broke_display_name"),
    ]

    for left, right in col_pairs:
        if left in df.columns and right in df.columns:
            left_norm = df[left].fillna("").astype(str).str.strip().str.casefold()
            right_norm = df[right].fillna("").astype(str).str.strip().str.casefold()

            mismatch = df[left_norm != right_norm]

            print(f"Checking columns: {left} vs {right}")
            print(f"  Total rows: {len(df)}")
            print(f"  Mismatches ignoring case: {len(mismatch)}")

            if not mismatch.empty:
                print(mismatch[[left, right]].head(20).to_string(index=False))
            return

    missing = [c for c in ("broker", "broker_display_name", "broke_display_name") if c not in df.columns]
    print("Missing required columns:", missing)


# Example usage:
compare_broker_columns(loaded_data)

Checking columns: broker vs broker_display_name
  Total rows: 3132
  Mismatches ignoring case: 0


In [16]:
# Rule:
# owner_association_activity_status should have a value only when broker_type == "Property Manager"

violations = loaded_data[
    (
        loaded_data["owner_association_activity_status"]
        .fillna("")
        .astype(str)
        .str.strip() != ""
    )
    &
    (
        loaded_data["broker_type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("Property Manager")
    )
]

print(f"Number of violations: {len(violations)}")

# Print relevant columns
print(
    violations[
        [
            "unique_id",
            "broker_type",
            "owner_association_activity_status"
        ]
    ]
)

Number of violations: 0
Empty DataFrame
Columns: [unique_id, broker_type, owner_association_activity_status]
Index: []


In [18]:
# Check price/currency consistency: if one is present, the other must be too
print("Checking price/currency consistency:\n")

if 'price' in loaded_data.columns and 'currency' in loaded_data.columns:
    # Identify rows with data in price vs currency
    has_price = loaded_data['price'].notna() & (loaded_data['price'].astype(str).str.strip() != '')
    has_currency = loaded_data['currency'].notna() & (loaded_data['currency'].astype(str).str.strip() != '')
    
    # Find violations: price XOR currency (one true, one false)
    violations = loaded_data[has_price != has_currency]
    
    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows violating rule: {len(violations)}")
    
    if len(violations) > 0:
        print("\nViolating row IDs:")
        violation_ids = violations['unique_id'].tolist() if 'unique_id' in loaded_data.columns else violations.index.tolist()
        print(violation_ids)
        
        print("\nSample violations:")
        cols_show = [c for c in ['unique_id', 'price', 'currency'] if c in loaded_data.columns]
        print(violations[cols_show].head(20).to_string(index=False))
    else:
        print("✓ All rows satisfy the rule: if price exists, currency exists and vice versa")
else:
    missing = []
    if 'price' not in loaded_data.columns:
        missing.append('price')
    if 'currency' not in loaded_data.columns:
        missing.append('currency')
    print(f"Missing columns: {missing}")

Checking price/currency consistency:

Total rows: 817
Rows violating rule: 0
✓ All rows satisfy the rule: if price exists, currency exists and vice versa


In [33]:
# Check if price fields use dot (.) instead of comma (,) as decimal separator
#price_columns = ['regular_price', 'selling_price', 'promotion_price', 'price_per_unit']
price_columns = ['regular_price','selling_price','price_was','promotion_price','price_per_unit']

print("Checking price field separators (should use dot '.' not comma ','):\n")

for col in price_columns:
    if col in loaded_data.columns:
        # Find rows with comma in price field
        comma_values = loaded_data[loaded_data[col].astype(str).str.contains(',', na=False)]
        
        if len(comma_values) > 0:
            print(f"❌ {col}: Found {len(comma_values)} values with COMMA separator")
            print(f"   Sample: {comma_values[col].head(3).tolist()}")
        else:
            print(f"✓ {col}: All values use dot (.) or are numeric")
    else:
        print(f"⚠ {col}: Column not found in data")
    print()

Checking price field separators (should use dot '.' not comma ','):

✓ regular_price: All values use dot (.) or are numeric

✓ selling_price: All values use dot (.) or are numeric

✓ price_was: All values use dot (.) or are numeric

✓ promotion_price: All values use dot (.) or are numeric

✓ price_per_unit: All values use dot (.) or are numeric



In [14]:

def is_valid(row):
    grammage_unit = str(row['grammage_unit']).strip().lower()
    site_shown_uom = str(row['site_shown_uom']).strip().lower()

    # Special case: stück can appear as St in site_shown_uom
    if grammage_unit == 'stück':
        return 'St' in site_shown_uom

    return grammage_unit in site_shown_uom

violations = loaded_data[~loaded_data.apply(is_valid, axis=1)]

print(
    violations[
        ['unique_id', 'grammage_quantity', 'grammage_unit', 'site_shown_uom']
    ]
)

           unique_id grammage_quantity grammage_unit site_shown_uom
1      2090000510157                 1         stück           1 St
5      4049639458550                 1         stück           1 St
8      4066447750843               240         stück         240 St
10     4066447707212                60         stück          60 St
11     4067796178791                30         stück          30 St
...              ...               ...           ...            ...
16640  3600530730308                 1         stück        1 stück
16644  3600524194925                 1         stück           1 St
16664  3574661900360                16         stück          16 St
16665  3574661776729                 1         stück           1 St
16666  3574661776644                 1         stück           1 St

[6226 rows x 4 columns]


In [14]:
# Convert to string to avoid errors with nulls/numbers
loaded_data['Part number'] = loaded_data['Part number'].astype(str)
loaded_data['URL'] = loaded_data['URL'].astype(str)

# Find rows where part number is NOT present in URL
violations = loaded_data[
    ~loaded_data.apply(lambda row: row['Part number'] in row['URL'], axis=1)
]

print("URLs violating the requirement:")
for url in violations['URL']:
    print(url)

URLs violating the requirement:


In [21]:
# Check if unique_id and pdp_url are not empty and unique
print("Validation Check: unique_id and pdp_url\n")
print("=" * 60)

# Check unique_id
print("\n📌 unique_id:")
unique_id_empty = loaded_data['unique_id'].isnull().sum() + (loaded_data['unique_id'].astype(str).str.strip() == '').sum()
unique_id_count = len(loaded_data['unique_id'].unique())
total_rows = len(loaded_data)

print(f"   Empty values: {unique_id_empty}")
print(f"   Total rows: {total_rows}")
print(f"   Unique values: {unique_id_count}")
print(f"   Is NOT empty: {'✓ YES' if unique_id_empty == 0 else '✗ NO'}")
print(f"   Is UNIQUE: {'✓ YES' if unique_id_count == total_rows else f'✗ NO ({total_rows - unique_id_count} duplicates)'}")

# Check pdp_url
print("\n📌 pdp_url:")
pdp_url_empty = loaded_data['pdp_url'].isnull().sum() + (loaded_data['pdp_url'].astype(str).str.strip() == '').sum()
pdp_url_count = len(loaded_data['pdp_url'].unique())

print(f"   Empty values: {pdp_url_empty}")
print(f"   Total rows: {total_rows}")
print(f"   Unique values: {pdp_url_count}")
print(f"   Is NOT empty: {'✓ YES' if pdp_url_empty == 0 else '✗ NO'}")
print(f"   Is UNIQUE: {'✓ YES' if pdp_url_count == total_rows else f'✗ NO ({total_rows - pdp_url_count} duplicates)'}")

print("\n" + "=" * 60)

Validation Check: unique_id and pdp_url


📌 unique_id:
   Empty values: 0
   Total rows: 11523
   Unique values: 11523
   Is NOT empty: ✓ YES
   Is UNIQUE: ✓ YES

📌 pdp_url:
   Empty values: 0
   Total rows: 11523
   Unique values: 11523
   Is NOT empty: ✓ YES
   Is UNIQUE: ✓ YES



In [7]:
# Validate grammage_quantity and grammage_unit
print("Grammage validation:\n")

valid_units = {
    'g', 'kg', 'mg', 'lb', 'oz', 'ml', 'l', 'ltr', 'litre', 'liter',
    'pcs', 'pc', 'pack', 'pkt', 'each', 'ea', 'count', 'unit', 'ct', 'kos', 'pz'
}

quantity_pattern = r'^\d+(\.\d+)?$'

if 'grammage_quantity' in loaded_data.columns:
    quantity = loaded_data['grammage_quantity'].astype(str).str.strip()
    invalid_quantity = loaded_data[~quantity.str.match(quantity_pattern, na=False)]
    invalid_comma = loaded_data[quantity.str.contains(',', na=False)]

    print(f"grammage_quantity rows with invalid format: {len(invalid_quantity)}")
    if len(invalid_quantity) > 0:
        print(invalid_quantity[['grammage_quantity']].head(10).to_dict('records'))

    print(f"grammage_quantity rows containing commas: {len(invalid_comma)}")
    if len(invalid_comma) > 0:
        print(invalid_comma[['grammage_quantity']].head(10).to_dict('records'))
else:
    print("grammage_quantity column not found.")

print()

if 'grammage_unit' in loaded_data.columns:
    unit = loaded_data['grammage_unit'].astype(str).str.strip().str.lower()
    invalid_unit = loaded_data[~unit.isin(valid_units) & ~loaded_data['grammage_unit'].isna()]

    print(f"grammage_unit rows with invalid units: {len(invalid_unit)}")
    if len(invalid_unit) > 0:
        print(invalid_unit[['grammage_unit']].head(10).to_dict('records'))
    else:
        print("grammage_unit values are valid.")
else:
    print("grammage_unit column not found.")

Grammage validation:

grammage_quantity rows with invalid format: 0
grammage_quantity rows containing commas: 0

grammage_unit rows with invalid units: 0
grammage_unit values are valid.


In [22]:
# Find duplicate pdp_url values
print("Duplicate pdp_url check\n")
print("=" * 60)

if 'pdp_url' in loaded_data.columns:
    url_series = loaded_data['pdp_url'].astype(str).str.strip()
    duplicate_mask = url_series.duplicated(keep=False)
    duplicate_rows = loaded_data.loc[duplicate_mask, ['unique_id', 'pdp_url']].copy()
    duplicate_counts = duplicate_rows['pdp_url'].value_counts()

    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows with duplicate pdp_url: {len(duplicate_rows)}")
    print(f"Unique duplicated pdp_url values: {len(duplicate_counts)}")
    print()
    print("Duplicate pdp_url counts:")
    print(duplicate_counts.head(20))

    if not duplicate_counts.empty:
        print("\nSample duplicate rows:")
        print(duplicate_rows.groupby('pdp_url').size().reset_index(name='count').sort_values('count', ascending=False).head(20).to_string(index=False))
else:
    print("pdp_url column not found.")

print("\n" + "=" * 60)

Duplicate pdp_url check

Total rows: 240727
Rows with duplicate pdp_url: 240449
Unique duplicated pdp_url values: 20528

Duplicate pdp_url counts:
pdp_url
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/572259/cynar-liquore-al-carciofo-e-erbe-70-cl                                 13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/159267/kelloggs-extra-granola-frutta-e-frutta-secca-500-g                     13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/109281/esselunga-bio-semi-di-zucca-tostati-200-g                              13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/551495/tenuta-rapitala-alcamo-vigna-casalj-75-cl                              13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/431006/smart-tonno-allolio-di-semi-di-girasole-3-x-80-g                       13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/755659/esselunga-bio-zuppa-di-verdure-

In [21]:
# Find duplicate pdp_url values
print("Duplicate unique_id check\n")
print("=" * 60)

if 'unique_id' in loaded_data.columns:
    id_series = loaded_data['unique_id'].astype(str).str.strip()
    duplicate_mask = id_series.duplicated(keep=False)
    duplicate_rows = loaded_data.loc[duplicate_mask, ['unique_id', 'pdp_url']].copy()
    duplicate_counts = duplicate_rows['pdp_url'].value_counts()

    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows with duplicate unique_id: {len(duplicate_rows)}")
    print(f"Unique duplicated unique_id values: {len(duplicate_counts)}")
    print()
    print("Duplicate unique_id counts:")
    print(duplicate_counts.head(20))

    if not duplicate_counts.empty:
        print("\nSample duplicate rows:")
        print(duplicate_rows.groupby('pdp_url').size().reset_index(name='count').sort_values('count', ascending=False).head(20).to_string(index=False))
else:
    print("unique_id column not found.")

print("\n" + "=" * 60)

Duplicate unique_id check

Total rows: 240727
Rows with duplicate unique_id: 240449
Unique duplicated unique_id values: 20528

Duplicate unique_id counts:
pdp_url
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/572259/cynar-liquore-al-carciofo-e-erbe-70-cl                                 13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/159267/kelloggs-extra-granola-frutta-e-frutta-secca-500-g                     13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/109281/esselunga-bio-semi-di-zucca-tostati-200-g                              13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/551495/tenuta-rapitala-alcamo-vigna-casalj-75-cl                              13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/431006/smart-tonno-allolio-di-semi-di-girasole-3-x-80-g                       13
https://spesaonline.esselunga.it/commerce/nav/drive/store/prodotto/755659/esselunga-bio-zuppa-di-

In [20]:
loaded_data.duplicated().sum()

0

In [23]:
# Validate that site_shown_uom ends with grammage_unit; print unique_id for mismatches
print("Checking site_shown_uom endswith grammage_unit:\n")

unit_aliases = {'ltr': 'l', 'litre': 'l', 'liter': 'l'}
mismatched_ids = []

if 'site_shown_uom' not in loaded_data.columns:
    print("site_shown_uom column not found. Cannot perform check.")
elif 'grammage_unit' not in loaded_data.columns:
    print("grammage_unit column not found. Cannot perform check.")
else:
    for idx, row in loaded_data.iterrows():
        site_shown_uom = row.get('site_shown_uom')
        unit = row.get('grammage_unit')
        uid = row.get('unique_id', idx)

        if pd.isna(site_shown_uom) or pd.isna(unit):
            continue

        site_shown_uom_s = str(site_shown_uom).strip().lower()
        unit_s = str(unit).strip().lower()
        unit_norm = unit_aliases.get(unit_s, unit_s)

        # valid end forms: ' <unit>' or '<unit>' (no space)
        candidates = [f" {unit_s}", unit_s, f" {unit_norm}", unit_norm]

        if not any(site_shown_uom_s.endswith(c) for c in candidates):
            mismatched_ids.append(uid)

    print(f"Total rows checked: {len(loaded_data)}")
    print(f"Rows where site_shown_uom does NOT end with grammage_unit: {len(mismatched_ids)}")
    if mismatched_ids:
        print("Sample unique_ids with mismatches:", mismatched_ids[:50])
        # show example rows
        cols_show = [c for c in ['unique_id','site_shown_uom','grammage_quantity','grammage_unit'] if c in loaded_data.columns]
        print("\nExample mismatched rows:")
        print(loaded_data[loaded_data['unique_id'].isin(mismatched_ids)][cols_show].head(10).to_dict('records'))

Checking site_shown_uom endswith grammage_unit:

Total rows checked: 11523
Rows where site_shown_uom does NOT end with grammage_unit: 204
Sample unique_ids with mismatches: [4009175968616, 9000101808506, 9000101566697, 4070765058802, 8006530170666, 4066447969870, 3830049641011, 4058172622410, 4066447792911, 4066447484519, 4066447792997, 4066447989922, 4066447369731, 4009175941787, 4066447989984, 5010622005029, 5000204254945, 3830049640274, 4066447888393, 4066447791181, 4066447791198, 4066447791495, 4066447791204, 4066447792935, 4066447791211, 7046110052457, 4066447791501, 4066447791228, 4066447791174, 9000101563603, 4066447793208, 4066447793222, 4066447792188, 4066447791235, 4066447792959, 4067796212853, 4066447951325, 4066447861297, 8700216722940, 4066447896886, 4070765022018, 4066447899436, 4066447231854, 9000101808445, 4066447896824, 4066447369755, 4066447877984, 9000101801255, 4058172172199, 4066447903584]

Example mismatched rows:
[{'unique_id': 4009175968616, 'site_shown_uom': '2

In [9]:
# Validate product_name + site_shown_uom rule
print("Checking product_name/site_shown_uom -> grammage fields:\n")

required_cols = {'product_name', 'site_shown_uom', 'grammage_quantity', 'grammage_unit'}
missing_cols = sorted(required_cols - set(loaded_data.columns))

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    def has_uom_in_product(row):
        product_name = str(row['product_name']).strip().lower() if pd.notna(row['product_name']) else ''
        site_uom = str(row['site_shown_uom']).strip().lower() if pd.notna(row['site_shown_uom']) else ''
        return bool(site_uom) and site_uom in product_name

    violations = loaded_data[
        loaded_data.apply(
            lambda row: has_uom_in_product(row)
            and (
                str(row['grammage_quantity']).strip() == ''
                or str(row['grammage_unit']).strip() == ''
            ),
            axis=1
        )
    ]

    print(f"Rows violating the rule: {len(violations)}")

    if not violations.empty:
        print(violations[['unique_id', 'product_name', 'site_shown_uom', 'grammage_quantity', 'grammage_unit']].to_string(index=False))
    else:
        print("No violations found.")


Checking product_name/site_shown_uom -> grammage fields:

Rows violating the rule: 0
No violations found.


In [ ]:
# Convert to string to avoid type issues
loaded_data['site_shown_uom'] = loaded_data['site_shown_uom'].astype(str)
loaded_data['grammage_quantity'] = loaded_data['grammage_quantity'].astype(str)

# Find rows where site_shown_uom does NOT end with grammage_quantity
violations = loaded_data[
    ~loaded_data['site_shown_uom'].str.endswith(loaded_data['grammage_quantity'], na=False)
]

print("Violating unique_ids:")
print(violations['unique_id'].tolist())

In [20]:
# Check rows where selling_price != regular_price
print("Price comparison: selling_price vs regular_price\n")

if 'selling_price' in loaded_data.columns and 'regular_price' in loaded_data.columns:
    mismatches = loaded_data[loaded_data['selling_price'] != loaded_data['regular_price']]
    
    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows where selling_price ≠ regular_price: {len(mismatches)}")
    
    if len(mismatches) > 0:
        mismatched_ids = mismatches['unique_id'].unique() if 'unique_id' in loaded_data.columns else mismatches.index.tolist()
        print(f"\nUnique IDs with price mismatch ({len(mismatched_ids)} unique):")
        print(list(mismatched_ids[:100]))  # Show first 100
        
        if len(mismatched_ids) > 100:
            print(f"... and {len(mismatched_ids) - 100} more")
        
        # Show sample rows
        cols_show = [c for c in ['unique_id', 'product_name', 'selling_price', 'regular_price'] if c in loaded_data.columns]
        print("\nSample mismatched rows:")
        print(mismatches[cols_show].head(10).to_dict('records'))
    else:
        print("✓ All rows have selling_price = regular_price")
else:
    if 'selling_price' not in loaded_data.columns:
        print("⚠ selling_price column not found.")
    if 'regular_price' not in loaded_data.columns:
        print("⚠ regular_price column not found.")

Price comparison: selling_price vs regular_price

Total rows: 40303
Rows where selling_price ≠ regular_price: 0
✓ All rows have selling_price = regular_price


In [19]:
# Check rows where regular_price != selling_price

violations = loaded_data[
    loaded_data["regular_price"].fillna("").astype(str).str.strip()
    !=
    loaded_data["selling_price"].fillna("").astype(str).str.strip()
]

if not violations.empty:
    print("Rows where regular_price != selling_price:")
    print(
        violations[
            ["unique_id", "pdp_url", "regular_price", "selling_price"]
        ].to_string(index=False)
    )
else:
    print("No violations found.")

No violations found.


In [29]:
# Check rows where promotion_description is empty but promotion_price has a value
print("Checking empty promotion description with promotion price:\n")

required_cols = {'promotion_description', 'promotion_price', 'unique_id'}
missing_cols = sorted(required_cols - set(loaded_data.columns))

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    violations = loaded_data[
        (loaded_data['promotion_description'].fillna('').astype(str).str.strip() == '')
        & (loaded_data['promotion_price'].fillna('').astype(str).str.strip() != '')
    ]

    print(f"Rows violating the rule: {len(violations)}")
    if not violations.empty:
        print("Violating unique_ids:")
        print(violations['unique_id'].astype(str).tolist())
    else:
        print("No violations found.")


Checking empty promotion description with promotion price:

Rows violating the rule: 0
No violations found.
